In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
import pandas as pd

from src.modeling import (
    prepare_train_test,
    scale_features,
    LogisticRegressionTrainer,
    RandomForestTrainer,
    GradientBoostingTrainer,
    XGBoostTrainer,
    ModelComparer,
    save_model
)


In [3]:
df = pd.read_csv(r"D:\GIT\data-analytics-portfolio\02-telco-customer-churn\data\processed\telco_customer_churn_model_data.csv")

df.head

<bound method NDFrame.head of       seniorcitizen  tenure  monthlycharges  totalcharges  \
0                 0       1           29.85         29.85   
1                 0      34           56.95       1889.50   
2                 0       2           53.85        108.15   
3                 0      45           42.30       1840.75   
4                 0       2           70.70        151.65   
...             ...     ...             ...           ...   
7016              0      24           84.80       1990.50   
7017              0      72          103.20       7362.90   
7018              0      11           29.60        346.45   
7019              1       4           74.40        306.60   
7020              0      66          105.65       6844.50   

      charge_per_month_tenure  num_additional_services  gender_Male  \
0                   14.925000                        1        False   
1                   53.985714                        2         True   
2                   36.0

In [4]:
df.shape

(7021, 39)

In [5]:
df.columns

Index(['seniorcitizen', 'tenure', 'monthlycharges', 'totalcharges',
       'charge_per_month_tenure', 'num_additional_services', 'gender_Male',
       'partner_Yes', 'dependents_Yes', 'phoneservice_Yes',
       'multiplelines_No phone service', 'multiplelines_Yes',
       'internetservice_Fiber optic', 'internetservice_No',
       'onlinesecurity_No internet service', 'onlinesecurity_Yes',
       'onlinebackup_No internet service', 'onlinebackup_Yes',
       'deviceprotection_No internet service', 'deviceprotection_Yes',
       'techsupport_No internet service', 'techsupport_Yes',
       'streamingtv_No internet service', 'streamingtv_Yes',
       'streamingmovies_No internet service', 'streamingmovies_Yes',
       'contract_One year', 'contract_Two year', 'paperlessbilling_Yes',
       'paymentmethod_Credit card (automatic)',
       'paymentmethod_Electronic check', 'paymentmethod_Mailed check', 'churn',
       'tenure_group_developing', 'tenure_group_established',
       'tenure_grou

In [6]:
df["churn"].value_counts()


churn
False    5164
True     1857
Name: count, dtype: int64

In [7]:
# Prepare features and target variables
X = df.drop("churn", axis=1)

y = df["churn"]

print(X.shape)
print(y.shape)


(7021, 38)
(7021,)


In [8]:
# Prepare train-test split
X_train, X_test, y_train, y_test = prepare_train_test(
    X,
    y
)



In [9]:
# Scale features
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)


(5616, 38)
(1405, 38)
(5616,)
(1405,)


In [10]:
# Scale features
X_train_scaled, X_test_scaled, scaler = scale_features(
    X_train,
    X_test
)
X_train_scaled.shape



(5616, 38)

In [11]:
# Baseline model: Logistic Regression

log_model = LogisticRegressionTrainer()

# Train the model

log_model.train(
    X_train_scaled,
    y_train
)

log_results = log_model.evaluate(
    X_test_scaled,
    y_test
)

log_results




{'Accuracy': 0.8078291814946619,
 'Precision': 0.6783216783216783,
 'Recall': 0.521505376344086,
 'F1 Score': 0.5896656534954408}

In [12]:
# Random Forest model

rf_model = RandomForestTrainer()

# Train the model
rf_model.train(
    X_train,
    y_train
)

# evaluate the model
rf_results = rf_model.evaluate(
    X_test,
    y_test
)

rf_results


{'Accuracy': 0.7914590747330961,
 'Precision': 0.6446886446886447,
 'Recall': 0.4731182795698925,
 'F1 Score': 0.5457364341085271}

In [13]:
# Gradient Boosting model

gb_model = GradientBoostingTrainer()

# train the model
gb_model.train(
    X_train,
    y_train
)
# evaluate the model

gb_results = gb_model.evaluate(
    X_test,
    y_test
)

gb_results



{'Accuracy': 0.799288256227758,
 'Precision': 0.6573426573426573,
 'Recall': 0.5053763440860215,
 'F1 Score': 0.5714285714285714}

In [14]:
# xgboost model

xgb_model = XGBoostTrainer()

# train the model
xgb_model.train(
    X_train,
    y_train
)

# evaluate the model
xgb_results = xgb_model.evaluate(
    X_test,
    y_test
)

xgb_results


{'Accuracy': 0.7843416370106762,
 'Precision': 0.6161616161616161,
 'Recall': 0.49193548387096775,
 'F1 Score': 0.547085201793722}

In [15]:
# model comparison
comparison = ModelComparer()

comparison.add_result(
    "Logistic Regression",
    log_results
)

comparison.add_result(
    "Random Forest",
    rf_results
)

comparison.add_result(
    "Gradient Boosting",
    gb_results
)

comparison.add_result(
    "XGBoost",
    xgb_results
)

# display the comparison results

comparison_df = comparison.get_results()

comparison_df


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.807829,0.678322,0.521505,0.589666
1,Random Forest,0.791459,0.644689,0.473118,0.545736
2,Gradient Boosting,0.799288,0.657343,0.505376,0.571429
3,XGBoost,0.784342,0.616162,0.491935,0.547085


In [16]:
comparison_df.sort_values(
    by="F1 Score",
    ascending=False
)


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.807829,0.678322,0.521505,0.589666
2,Gradient Boosting,0.799288,0.657343,0.505376,0.571429
3,XGBoost,0.784342,0.616162,0.491935,0.547085
1,Random Forest,0.791459,0.644689,0.473118,0.545736


In [17]:
best_model_name = comparison_df.sort_values(
    by="F1 Score",
    ascending=False
).iloc[0]

best_model_name


Model        Logistic Regression
Accuracy                0.807829
Precision               0.678322
Recall                  0.521505
F1 Score                0.589666
Name: 0, dtype: object

### Model Comparison Summary


#### 1. Which model achieved the highest F1 Score?
**Logistic Regression** achieved the highest F1 Score of **0.590**, indicating the best balance between Precision and Recall.

#### 2. Which model achieved the highest Recall?
**Logistic Regression** achieved the highest Recall of **0.522**, meaning it identified the largest proportion of churned customers.

#### 3. Which metric is most important for churn prediction?
**Recall** is often the most important metric for churn prediction because missing a customer who is likely to churn can result in lost revenue and missed retention opportunities.

#### 4. Why might a model with lower accuracy still be preferred?
A model with lower Accuracy may be preferred if it has a higher Recall or F1 Score. In churn prediction, correctly identifying customers who are likely to leave is usually more valuable than achieving the highest overall Accuracy, especially when the classes are imbalanced.


### Conclusion 

- **Logistic Regression** achieved the best overall performance with the highest Accuracy (**80.8%**) and F1 Score (**0.590**), making it the preferred model for churn prediction.
- **Gradient Boosting** ranked second with an F1 Score of **0.571**, showing competitive performance but slightly lower Recall than Logistic Regression.
- **Random Forest** and **XGBoost** produced similar results, with F1 Scores of **0.546** and **0.547** respectively, indicating weaker performance in identifying churned customers.
- Overall, **Logistic Regression provided the best balance between Precision and Recall**, making it the most suitable model for this dataset.

**Selected Model:** Logistic Regression 

In [20]:
# Save Logistic Regression

save_model(
    log_model.model,
    r"D:\GIT\data-analytics-portfolio\02-telco-customer-churn\models\logistic_regression.pkl"
)

# Save Random Forest

save_model(
    rf_model.model,
    r"D:\GIT\data-analytics-portfolio\02-telco-customer-churn\models\random_forest.pkl"
)

# Save Gradient Boosting

save_model(
    gb_model.model,
    r"D:\GIT\data-analytics-portfolio\02-telco-customer-churn\models\gradient_boosting.pkl"
)

# Save XGBoost (if trained)

try:
    save_model(
        xgb_model.model,
        r"D:\GIT\data-analytics-portfolio\02-telco-customer-churn\models\xgboost.pkl"
    )
except NameError:
    print("XGBoost model was not trained.")

# Save the scaler
save_model(
    scaler,
    r"D:\GIT\data-analytics-portfolio\02-telco-customer-churn\models\scaler.pkl"
)
    


In [22]:
import os

sorted(os.listdir(r"D:\GIT\data-analytics-portfolio\02-telco-customer-churn\models"))


['gradient_boosting.pkl',
 'logistic_regression.pkl',
 'random_forest.pkl',
 'scaler.pkl',
 'xgboost.pkl']